# Search Videos

Find specific moments, topics, or content across your videos and images with natural language.
Use this for locating content in a large collection, matching moments to a description, or building search features in your app.

# Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

In [ ]:
import json
import os

from twelvelabs import TwelveLabs, TextParam
from twelvelabs.types.text_param_format import TextParamFormat_JsonSchema

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "<YOUR_API_KEY>")
STORE_ID = os.environ.get("TWELVELABS_STORE_ID", "<YOUR_KNOWLEDGE_STORE_ID>")  # Replace with your knowledge store ID

client = TwelveLabs(api_key=API_KEY)

## Helper Functions

A utility to extract text content from a Jockey API response.

In [ ]:
def parse_response(response) -> str:
    """Extract text content from a Jockey response.

    Args:
        response: The ResponseObject returned by client.responses.create().

    Returns:
        The text content from the first message output, or an empty string
        if no message content is found.
    """
    for output in response.output:
        if output.type == "message":
            for content in output.content:
                return content.text
    return ""

## Search Schema

Define a JSON schema for structured search results. Each result includes an item reference,
timestamp, description, and relevance score. The schema also captures total result count
and how Jockey interpreted your query.

In [ ]:
SEARCH_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "item_reference": {"type": "string"},
                    "timestamp": {"type": "string"},
                    "description": {"type": "string"},
                    "relevance": {"type": "string"},
                },
            },
        },
        "total_results": {"type": "integer"},
        "query_interpretation": {"type": "string"},
    },
}

## Run a Search

Send a natural-language query to find matching moments across all videos and images in your knowledge store.
The structured response makes it easy to iterate over results programmatically.

In [ ]:
SEARCH_QUERY = "Find all moments where someone is presenting to an audience"

response = client.responses.create(
    knowledge_store_id=STORE_ID,
    input=[
        {"type": "message", "role": "user", "content": SEARCH_QUERY}
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="search_results", schema_=SEARCH_SCHEMA)
    ),
)

search = json.loads(parse_response(response))

print(f"Found {search['total_results']} results")
print(f"Interpreted as: {search['query_interpretation']}\n")

for r in search["results"]:
    print(f"  [{r['timestamp']}] {r['item_reference']}")
    print(f"    {r['description']}")
    print(f"    Relevance: {r['relevance']}\n")

## Example Queries

| Query | What It Finds |
|-------|---------------|
| "someone laughing" | Moments with laughter |
| "product being held up to camera" | Product showcase moments |
| "outdoor scenes with water" | Nature/water visuals |
| "heated discussion" | Tense conversational moments |
| "text on screen" | Moments with overlaid text or titles |

## Variations

- **Narrow by context:** Add instructions like "Only search the first 2 minutes of each video"
- **Ranked results:** "Find and rank the top 5 most visually striking moments"
- **Multi-turn refinement:** Search, then follow up with "Show me more like the third result"

## Next Steps

- **[Get Corpus Overview](get_corpus_overview.ipynb)** -- understand what's in your collection first
- **[Extract Entities](extract_entities.ipynb)** -- list all people, places, objects, and concepts
- **[Find Organization Axes](find_organization_axes.ipynb)** -- discover the best categorization strategies
- **[Enrich Content](enrich_content.ipynb)** -- get deeper, domain-specific analysis

See also:
- [Querying Guide](https://docs.twelvelabs.io/v1.3/agents/guides/create-a-response) -- fundamentals of the Responses API